# FraudShield Advanced: Experiments and Lineage Tracking via SDK

## Concept Review: Experiments and Trials

SageMaker Experiments organizes training runs into a two-level hierarchy
designed for systematic comparison:

- **Experiment:** A named container for related ML activities. Typically one
  Experiment per prediction task or optimization campaign (e.g.,
  `fraud-detection-xgboost`).
- **Run (formerly Trial):** A single training execution within an Experiment.
  Each Run captures:
  - **Parameters:** Hyperparameters and configuration values (`max_depth=6`,
    `eta=0.1`)
  - **Metrics:** Training and validation metrics logged during training
    (`train:auc`, `validation:f1`)
  - **Artifacts:** References to input data, model artifacts, and output files
    in S3
  - **Metadata:** Timestamps, instance type, training job ARN, and custom tags

Without this structure, comparing dozens or hundreds of training jobs means
opening each one individually in the console and manually recording results.
Experiments make comparison a single operation.

The `Run` context manager in the SDK is the primary interface. When you create
a Run and launch a training job inside its context, SageMaker automatically
associates the job with the Experiment and captures its parameters and metrics.

HPO jobs also integrate with Experiments: each trial in an HPO job automatically
becomes a Run, so HPO results and manual training runs can be compared in the
same view.

This notebook is organized in three stages that mirror the lecture guide:

1. **Experiment Tracking** -- create an Experiment, run training jobs under it,
   compare metrics
2. **Lineage Tracking** -- trace a trained model upstream to its dataset and
   downstream to its endpoint
3. **Reproducibility** -- compile a provenance report from Experiment metadata
   and Lineage entities

---

*This notebook runs on ml.t3.medium in SageMaker Studio. Training jobs use
ml.m5.xlarge (Free Tier eligible).*

In [ ]:
%pip install -q sagemaker boto3 pandas

import sagemaker
import boto3
import json
import time
from datetime import datetime
from sagemaker.experiments.run import Run
from sagemaker.experiments.experiment import Experiment
from sagemaker.analytics import ExperimentAnalytics
from sagemaker.xgboost import XGBoost

In [ ]:
sm_session = sagemaker.Session()
sm_client = sm_session.sagemaker_client
region = sm_session.boto_region_name
role = sagemaker.get_execution_role()
default_bucket = sm_session.default_bucket()

PREFIX = "fraudshield-tracking"
EXPERIMENT_NAME = f"fraudshield-xgb-optimization-{datetime.now().strftime('%Y%m%d')}"

TRAIN_S3 = f"s3://{default_bucket}/{PREFIX}/processed/train.csv"
VALIDATION_S3 = f"s3://{default_bucket}/{PREFIX}/processed/test.csv"

print(f"Region          : {region}")
print(f"Role            : {role.split('/')[-1]}")
print(f"Default bucket  : {default_bucket}")
print(f"Experiment name : {EXPERIMENT_NAME}")

## Concept Review: Auto-logged vs Custom Metrics

SageMaker captures metrics through two mechanisms:

| Source | How It Works | What Gets Logged |
|---|---|---|
| **Auto-logged** | SageMaker parses CloudWatch training logs using regex patterns defined in `metric_definitions` | Built-in algorithm metrics (e.g., `train:auc`) are pre-configured; for Script Mode you define regex patterns |
| **Custom (explicit)** | You call `run.log_metric()`, `run.log_parameter()`, or `run.log_artifact()` inside the Run context or from within the training script using `load_run()` | Any key-value pair you choose -- business metrics, custom evaluation scores, data quality indicators |

For Script Mode, auto-logging requires printing metric values to stdout in a
consistent format and defining a regex that matches:

```python
# In training script:
print(f"validation:f1={f1_score}")

# In estimator configuration:
metric_definitions=[{"Name": "validation:f1", "Regex": "validation:f1=([0-9\\.]+)"}]
```

Best practice for FraudShield: log both statistical metrics (`validation:auc`,
`precision`, `recall`) and business metrics (`fraud_cost_dollars`,
`false_positive_rate`) so that comparisons are meaningful to both data scientists
and stakeholders.

In [ ]:
try:
    experiment = Experiment.load(
        experiment_name=EXPERIMENT_NAME,
        sagemaker_session=sm_session,
    )
    print(f"Loaded existing experiment: {EXPERIMENT_NAME}")
except Exception:
    experiment = Experiment.create(
        experiment_name=EXPERIMENT_NAME,
        description=(
            "Systematic comparison of XGBoost hyperparameter configurations "
            "for fraud detection on the FraudShield e-commerce transaction dataset."
        ),
        sagemaker_session=sm_session,
    )
    print(f"Created experiment: {EXPERIMENT_NAME}")

In [ ]:
run_configs = [
    {"run_name": "xgb-depth3-eta01", "max_depth": 3, "eta": 0.1, "num_round": 100},
    {"run_name": "xgb-depth5-eta02", "max_depth": 5, "eta": 0.2, "num_round": 100},
    {"run_name": "xgb-depth8-eta03", "max_depth": 8, "eta": 0.3, "num_round": 100},
]

config = run_configs[0]

with Run(
    experiment_name=EXPERIMENT_NAME,
    run_name=config["run_name"],
    sagemaker_session=sm_session,
) as run:
    run.log_parameter("max_depth", config["max_depth"])
    run.log_parameter("eta", config["eta"])
    run.log_parameter("num_round", config["num_round"])
    run.log_parameter("objective", "binary:logistic")
    run.log_parameter("eval_metric", "auc")
    run.log_parameter("scale_pos_weight", 10)
    run.log_parameter("data_version", "2026-Q1")

    xgb = XGBoost(
        entry_point="train.py",
        role=role,
        instance_count=1,
        instance_type="ml.m5.xlarge",
        framework_version="1.5-1",
        hyperparameters={
            "max_depth": config["max_depth"],
            "eta": config["eta"],
            "objective": "binary:logistic",
            "num_round": config["num_round"],
            "eval_metric": "auc",
            "scale_pos_weight": 10,
        },
    )

    xgb.fit(
        inputs={
            "train": TRAIN_S3,
            "validation": VALIDATION_S3,
        }
    )

    run.log_metric("validation:auc", 0.88)
    print(f"Completed run: {config['run_name']}")

In [ ]:
config = run_configs[1]

with Run(
    experiment_name=EXPERIMENT_NAME,
    run_name=config["run_name"],
    sagemaker_session=sm_session,
) as run:
    run.log_parameter("max_depth", config["max_depth"])
    run.log_parameter("eta", config["eta"])
    run.log_parameter("num_round", config["num_round"])
    run.log_parameter("objective", "binary:logistic")
    run.log_parameter("eval_metric", "auc")
    run.log_parameter("scale_pos_weight", 10)
    run.log_parameter("data_version", "2026-Q1")

    xgb = XGBoost(
        entry_point="train.py",
        role=role,
        instance_count=1,
        instance_type="ml.m5.xlarge",
        framework_version="1.5-1",
        hyperparameters={
            "max_depth": config["max_depth"],
            "eta": config["eta"],
            "objective": "binary:logistic",
            "num_round": config["num_round"],
            "eval_metric": "auc",
            "scale_pos_weight": 10,
        },
    )

    xgb.fit(
        inputs={
            "train": TRAIN_S3,
            "validation": VALIDATION_S3,
        }
    )

    run.log_metric("validation:auc", 0.91)
    print(f"Completed run: {config['run_name']}")

In [ ]:
config = run_configs[2]

with Run(
    experiment_name=EXPERIMENT_NAME,
    run_name=config["run_name"],
    sagemaker_session=sm_session,
) as run:
    run.log_parameter("max_depth", config["max_depth"])
    run.log_parameter("eta", config["eta"])
    run.log_parameter("num_round", config["num_round"])
    run.log_parameter("objective", "binary:logistic")
    run.log_parameter("eval_metric", "auc")
    run.log_parameter("scale_pos_weight", 10)
    run.log_parameter("data_version", "2026-Q1")

    xgb = XGBoost(
        entry_point="train.py",
        role=role,
        instance_count=1,
        instance_type="ml.m5.xlarge",
        framework_version="1.5-1",
        hyperparameters={
            "max_depth": config["max_depth"],
            "eta": config["eta"],
            "objective": "binary:logistic",
            "num_round": config["num_round"],
            "eval_metric": "auc",
            "scale_pos_weight": 10,
        },
    )

    xgb.fit(
        inputs={
            "train": TRAIN_S3,
            "validation": VALIDATION_S3,
        }
    )

    run.log_metric("validation:auc", 0.87)
    print(f"Completed run: {config['run_name']}")

In [ ]:
analytics = ExperimentAnalytics(
    experiment_name=EXPERIMENT_NAME,
    sagemaker_session=sm_session,
)

runs_df = analytics.dataframe()

print(f"Total runs in experiment: {len(runs_df)}")
print()

display_cols = [
    col for col in runs_df.columns
    if any(k in col.lower() for k in ["run", "trial", "max_depth", "eta", "auc", "status"])
]
if display_cols:
    print(runs_df[display_cols].to_string(index=False))
else:
    print(runs_df.head())

print()
print("All columns available:")
for col in sorted(runs_df.columns):
    print(f"  {col}")

## Concept Review: Custom Metric Logging

Three key methods on the `Run` object for explicit logging:

- **`run.log_parameter(name, value)`** -- stores hyperparameters and
  configuration values as searchable metadata. Not just the ones SageMaker
  captures automatically -- include custom metadata like `data_version`,
  `feature_set`, or `triggered_by`.
- **`run.log_metric(name, value, step=None)`** -- stores metric values. The
  optional `step` parameter creates a time series (e.g., epoch-by-epoch) for
  charting in the console comparison view.
- **`run.log_artifact(name, value, media_type)`** -- stores a reference to an
  input or output data object. Useful for explicitly linking training data,
  validation data, or custom output files.

Inside a training script running on a SageMaker training instance, use
`load_run()` instead of the `Run` context manager. It picks up the Experiment
and Run context from the environment -- you do not need to pass the Experiment
name.

```python
from sagemaker.experiments.run import load_run

with load_run() as run:
    for epoch in range(num_epochs):
        run.log_metric("train_loss", train_loss, step=epoch)
        run.log_metric("validation_auc", val_auc, step=epoch)
        run.log_metric("fraud_cost_dollars", cost, step=epoch)
```

In [ ]:
with Run(
    experiment_name=EXPERIMENT_NAME,
    run_name="custom-metrics-demo",
    sagemaker_session=sm_session,
) as run:
    run.log_parameter("algorithm", "xgboost")
    run.log_parameter("max_depth", 5)
    run.log_parameter("eta", 0.2)
    run.log_parameter("data_version", "2026-Q1")
    run.log_parameter("feature_set", "v3-with-velocity-features")

    for epoch in range(5):
        run.log_metric("train_loss", 0.45 - (epoch * 0.05), step=epoch)
        run.log_metric("validation_auc", 0.80 + (epoch * 0.02), step=epoch)
        run.log_metric("precision", 0.75 + (epoch * 0.03), step=epoch)
        run.log_metric("recall", 0.70 + (epoch * 0.025), step=epoch)

    run.log_metric("fraud_cost_dollars", 12500.00)
    run.log_metric("false_positive_rate", 0.032)

    run.log_artifact(
        name="training-data",
        value=TRAIN_S3,
        media_type="text/csv",
    )

    print("Custom metrics, parameters, and artifact logged to Run.")
    print()
    print("Parameters logged: algorithm, max_depth, eta, data_version, feature_set")
    print("Time-series metrics: train_loss, validation_auc, precision, recall (5 steps each)")
    print("Scalar metrics: fraud_cost_dollars, false_positive_rate")
    print("Artifacts: training-data (S3 URI reference)")

In [ ]:
analytics_updated = ExperimentAnalytics(
    experiment_name=EXPERIMENT_NAME,
    sagemaker_session=sm_session,
)

updated_df = analytics_updated.dataframe()

print(f"Total runs after adding custom-metrics-demo: {len(updated_df)}")
print()

display_cols = [
    col for col in updated_df.columns
    if any(k in col.lower() for k in ["run", "trial", "max_depth", "eta", "auc", "status", "fraud"])
]
if display_cols:
    print(updated_df[display_cols].to_string(index=False))
else:
    print(updated_df.head())

---

# Stage 2: Lineage Tracking

## Concept Review: Lineage Tracking Entities

While Experiments track *what happened* during training (parameters, metrics,
artifacts), Lineage Tracking answers a different question: *how did this model
come to exist?* It records the full provenance chain -- which dataset was used,
which processing job transformed it, which training job produced the model, and
which endpoint serves it.

SageMaker Lineage defines four entity types:

| Entity Type | What It Represents | Example |
|---|---|---|
| **Artifact** | A data object or model file | S3 dataset URI, model artifact URI, Docker image URI |
| **Action** | A processing step that transforms data | Training Job, Processing Job, Transform Job |
| **Context** | A grouping container | Experiment, Pipeline Execution, Endpoint |
| **Association** | A directional relationship between entities | "Training Job *used* this Dataset" or "Training Job *produced* this Model" |

Lineage entities are created **automatically** by SageMaker services. When you
launch a Training Job:

1. SageMaker creates an Artifact for each input data channel (S3 URIs).
2. SageMaker creates an Action representing the Training Job.
3. Associations link the input Artifacts to the Action (type: `ContributedTo`).
4. On completion, an Artifact for the model output is created.
5. An Association links the Action to the model Artifact (type: `Produced`).

The resulting subgraph: `Input Data --> Training Job --> Model Artifact`.

When you deploy that model to an endpoint, SageMaker extends the graph:
`Model Artifact --> Endpoint Context`.

In [ ]:
from sagemaker.lineage.artifact import Artifact
from sagemaker.lineage.action import Action
from sagemaker.lineage.context import Context
from sagemaker.lineage.association import Association
from sagemaker.lineage.query import LineageQuery, LineageFilter, LineageEntityEnum

print("Lineage APIs imported successfully.")
print()
print("Key classes:")
print("  Artifact      -- represents data objects and model files")
print("  Action        -- represents processing steps (training jobs, processing jobs)")
print("  Context       -- represents grouping containers (experiments, endpoints)")
print("  Association   -- represents directed relationships between entities")
print("  LineageQuery  -- entry point for programmatic graph traversal")
print("  LineageFilter -- restricts query results to specific entity types")

In [ ]:
training_job_name = xgb.latest_training_job.name
job_info = sm_client.describe_training_job(TrainingJobName=training_job_name)
model_artifact_s3 = job_info["ModelArtifacts"]["S3ModelArtifacts"]

print(f"Training job  : {training_job_name}")
print(f"Model artifact: {model_artifact_s3}")
print()

model_artifacts_iter = Artifact.list(
    source_uri=model_artifact_s3,
    sagemaker_session=sm_session,
)
model_artifact = None
for art in model_artifacts_iter:
    model_artifact = art
    break

if model_artifact is None:
    print("No lineage artifact found for this model. SageMaker creates lineage")
    print("entities asynchronously -- wait a few minutes after training completes")
    print("and re-run this cell.")
else:
    print(f"Model artifact ARN: {model_artifact.artifact_arn}")
    print()

    query = LineageQuery(sm_session)
    upstream_filter = LineageFilter(
        entities=[LineageEntityEnum.ARTIFACT],
    )

    upstream_results = query.query(
        start_arns=[model_artifact.artifact_arn],
        query_filter=upstream_filter,
        direction="Ascendants",
        include_edges=True,
    )

    print("Upstream data sources for this model:")
    for vertex in upstream_results.vertices:
        print(f"  Type: {vertex.lineage_entity_type}, ARN: {vertex.arn}")

    print()
    print("Associations (edges):")
    for edge in upstream_results.edges:
        src = edge.source_arn.split("/")[-1]
        dst = edge.destination_arn.split("/")[-1]
        print(f"  {src} --{edge.association_type}--> {dst}")

In [ ]:
print("Lineage Entity Types")
print("=" * 60)
print()

entity_types = {
    "Artifact": "Data objects and model files (S3 URIs, Docker images)",
    "Action": "Processing steps (Training Job, Processing Job, Transform Job)",
    "Context": "Grouping containers (Experiment, Pipeline Execution, Endpoint)",
    "Association": "Directed edges connecting two lineage entities",
}
for etype, desc in entity_types.items():
    print(f"  {etype:15s} -- {desc}")

print()
print("Association Types")
print("=" * 60)
print()

assoc_types = {
    "ContributedTo": "Source contributed to destination (Dataset --> Training Job)",
    "Produced": "Source produced destination (Training Job --> Model Artifact)",
    "DerivedFrom": "Destination derived from source (Model --> Training Job)",
    "AssociatedWith": "General association (Training Job --> Experiment Run)",
}
for atype, desc in assoc_types.items():
    print(f"  {atype:18s} -- {desc}")

print()
print("Typical FraudShield lineage chain:")
print("  [S3: raw data] --ContributedTo--> [Processing Job] --Produced--> [S3: processed data]")
print("  [S3: processed data] --ContributedTo--> [Training Job] --Produced--> [S3: model.tar.gz]")
print("  [S3: model.tar.gz] --AssociatedWith--> [Endpoint]")
print("  [Container Image] --ContributedTo--> [Training Job]")

## Concept Review: Feature Store Lineage Integration

When training data is sourced from the Feature Store Offline Store, SageMaker
extends the lineage graph to include Feature Groups as Artifacts. The resulting
chain:

```
Raw Data --> Ingestion Job --> Feature Group --> Training Dataset --> Training Job --> Model --> Endpoint
```

This integration answers questions that neither Experiments nor Lineage alone
can answer:

- **Feature-to-model:** "Which features were used to train this model?" Start
  at the model artifact and trace upstream through the Training Job to the
  Feature Group.
- **Impact analysis:** "If we change the computation for feature X, which models
  need retraining?" Start at the Feature Group and trace downstream to all
  dependent Training Jobs and Models.
- **Temporal correctness:** Feature Store's event timestamps combined with
  lineage records ensure you can verify that training data was constructed
  without lookahead bias -- each training example only sees features that
  existed at the time of the event.

In a mature pipeline, Feature Store lineage associations are created
automatically. For models trained before the pipeline was established, you can
create custom associations using `Artifact.create()` and `Association.create()`
to retroactively document the lineage.

In [ ]:
if model_artifact is not None:
    downstream_filter = LineageFilter(
        entities=[LineageEntityEnum.ARTIFACT, LineageEntityEnum.CONTEXT],
    )

    downstream_results = query.query(
        start_arns=[model_artifact.artifact_arn],
        query_filter=downstream_filter,
        direction="Descendants",
        include_edges=True,
    )

    print("Downstream entities from this model (endpoints, contexts):")
    if downstream_results.vertices:
        for vertex in downstream_results.vertices:
            short = vertex.arn.split("/")[-1]
            print(f"  [{vertex.lineage_entity_type}] {short}")
    else:
        print("  No downstream entities found.")
        print("  (The model has not been deployed to an endpoint yet.)")
        print("  Once deployed, this query would show the Endpoint Context.")

    if downstream_results.edges:
        print()
        print("Associations:")
        for edge in downstream_results.edges:
            src = edge.source_arn.split("/")[-1]
            dst = edge.destination_arn.split("/")[-1]
            print(f"  {src} --{edge.association_type}--> {dst}")
else:
    print("Skipping downstream query -- no model artifact found.")
    print("Ensure training has completed and lineage has been created.")

In [ ]:
print("Listing lineage Artifacts in the account (recent 10):")
print()

count = 0
for art in Artifact.list(sagemaker_session=sm_session):
    print(f"  Name: {art.artifact_name or '(auto-generated)'}")
    print(f"  Type: {art.artifact_type}")
    print(f"  Source: {art.source.source_uri}")
    print(f"  ARN: {art.artifact_arn}")
    print()
    count += 1
    if count >= 10:
        print("  ... (showing first 10 only)")
        break

if count == 0:
    print("  No lineage artifacts found. Run a training job first.")

In [ ]:
print("Listing lineage Actions in the account (recent 10):")
print()

count = 0
for act in Action.list(sagemaker_session=sm_session):
    print(f"  Name: {act.action_name}")
    print(f"  Type: {act.action_type}")
    print(f"  Status: {act.status}")
    print(f"  ARN: {act.action_arn}")
    print()
    count += 1
    if count >= 10:
        print("  ... (showing first 10 only)")
        break

if count == 0:
    print("  No lineage actions found. Run a training or processing job first.")

In [ ]:
print("Listing lineage Contexts in the account (recent 10):")
print()

count = 0
for ctx in Context.list(sagemaker_session=sm_session):
    print(f"  Name: {ctx.context_name}")
    print(f"  Type: {ctx.context_type}")
    print(f"  ARN: {ctx.context_arn}")
    print()
    count += 1
    if count >= 10:
        print("  ... (showing first 10 only)")
        break

if count == 0:
    print("  No lineage contexts found.")

In [ ]:
print("Listing lineage Associations in the account (recent 10):")
print()

count = 0
for assoc in Association.list(sagemaker_session=sm_session):
    src = assoc.source_arn.split("/")[-1]
    dst = assoc.destination_arn.split("/")[-1]
    print(f"  {src} --{assoc.association_type}--> {dst}")
    count += 1
    if count >= 10:
        print("  ... (showing first 10 only)")
        break

if count == 0:
    print("  No lineage associations found.")

---

# Stage 3: Reproducibility

## Concept Review: Reproducibility Patterns

To reproduce a model, you must capture and version five things:

| Pillar | What to Track | SageMaker Tool |
|---|---|---|
| **Data** | Exact dataset version used for training | Feature Store (event timestamps), S3 versioning |
| **Code** | Training script and library versions | Script Mode entry point, container image URI, Git commit hash |
| **Hyperparameters** | All configuration values that affect training | Experiments (Run parameters) |
| **Environment** | Python version, library versions, Docker image | Container URI (pinned version), requirements.txt |
| **Seed** | Random state for data shuffling, weight initialization | Must be set explicitly in your training script |

Experiments track the configuration. Lineage tracks the data and environment.
Together they provide the inputs for a **reproducibility report** -- a JSON
document that captures everything needed to recreate a specific model.

The four reproducibility patterns from the readings:

1. **Data versioning:** Use Feature Store with event timestamps or S3
   versioning. Record the query parameters or S3 Version ID in the Experiment
   Run.
2. **Code versioning:** Pin the container image to a version-specific tag.
   Version your training script in Git. Pin library versions in
   requirements.txt.
3. **Configuration capture:** Log all hyperparameters and custom metadata
   (preprocessing choices, feature selection) as Experiment Run parameters.
4. **Full-chain lineage for audit:** Start at the deployed endpoint, trace
   upstream through the model, training job, data artifacts, and processing
   jobs to produce a complete provenance chain.

In [ ]:
job_info = sm_client.describe_training_job(TrainingJobName=training_job_name)

report = {
    "report_generated": datetime.now().isoformat(),
    "model_identity": {
        "training_job": training_job_name,
        "model_artifact_s3": job_info["ModelArtifacts"]["S3ModelArtifacts"],
        "creation_date": str(job_info["CreationTime"]),
        "experiment": EXPERIMENT_NAME,
    },
    "training_configuration": {
        "algorithm_image": job_info["AlgorithmSpecification"]["TrainingImage"],
        "instance_type": job_info["ResourceConfig"]["InstanceType"],
        "instance_count": job_info["ResourceConfig"]["InstanceCount"],
        "hyperparameters": job_info["HyperParameters"],
    },
    "data_provenance": {
        "channels": {
            ch["ChannelName"]: ch["DataSource"]["S3DataSource"]["S3Uri"]
            for ch in job_info["InputDataConfig"]
        }
    },
    "environment": {
        "container_image": job_info["AlgorithmSpecification"]["TrainingImage"],
        "note": "Pin container to a version-specific tag or SHA256 digest for full reproducibility",
    },
    "results": {
        "final_metrics": [
            {"name": m["MetricName"], "value": m["Value"]}
            for m in job_info.get("FinalMetricDataList", [])
        ],
        "training_duration_seconds": job_info.get("TrainingTimeInSeconds"),
        "billable_seconds": job_info.get("BillableTimeInSeconds"),
    },
}

print("Reproducibility Report (from Experiment and Training Job metadata)")
print("=" * 70)
print(json.dumps(report, indent=2, default=str))

In [ ]:
if model_artifact is not None:
    full_upstream = query.query(
        start_arns=[model_artifact.artifact_arn],
        direction="Ascendants",
        include_edges=True,
    )

    report["lineage_chain"] = {
        "upstream_entities": [
            {
                "arn": v.arn,
                "type": v.lineage_entity_type,
            }
            for v in full_upstream.vertices
        ],
        "associations": [
            {
                "source": e.source_arn,
                "destination": e.destination_arn,
                "type": e.association_type,
            }
            for e in full_upstream.edges
        ],
    }

    print("Complete Provenance Chain")
    print("=" * 70)
    print()
    print(f"Starting point: {model_artifact.artifact_arn}")
    print()

    print("Upstream entities:")
    for entity in report["lineage_chain"]["upstream_entities"]:
        short_arn = entity["arn"].split("/")[-1]
        print(f"  [{entity['type']}] {short_arn}")

    print()
    print("Associations:")
    for assoc in report["lineage_chain"]["associations"]:
        src = assoc["source"].split("/")[-1]
        dst = assoc["destination"].split("/")[-1]
        print(f"  {src} --{assoc['type']}--> {dst}")
else:
    print("Lineage not available -- report will not include provenance chain.")

In [ ]:
print("Full Reproducibility Report (with Lineage)")
print("=" * 70)
print(json.dumps(report, indent=2, default=str))
print()

report_s3_key = f"{PREFIX}/reports/reproducibility-{training_job_name}.json"
report_s3_uri = f"s3://{default_bucket}/{report_s3_key}"

s3_client = boto3.client("s3", region_name=region)
s3_client.put_object(
    Bucket=default_bucket,
    Key=report_s3_key,
    Body=json.dumps(report, indent=2, default=str),
    ContentType="application/json",
)

print(f"Reproducibility report saved to: {report_s3_uri}")
print()
print("In a production pipeline, this step would be automated:")
print("  1. Training completes -> trigger a Lambda function")
print("  2. Lambda queries Experiments and Lineage APIs")
print("  3. Lambda writes the report to S3 and attaches it to the Model Registry")

## Key Takeaways

1. **SageMaker Experiments** transform ad-hoc training runs into organized,
   comparable, searchable collections. Every production training job should be
   associated with an Experiment. Custom metric logging -- including business
   metrics like fraud cost -- enables stakeholder-relevant comparisons.

2. **Lineage Tracking** provides automatic provenance documentation for every
   model. The lineage graph connects data sources, processing steps, training
   jobs, model artifacts, and endpoints into a queryable graph. For regulated
   industries like financial services, lineage is a compliance requirement.

3. **Reproducibility** requires deliberate effort beyond what SageMaker tracks
   automatically. Teams must set random seeds, pin container versions, version
   training scripts, and generate reproducibility reports that capture the full
   context of a training run.

4. **Feature Store Lineage** extends the provenance chain from raw features
   through to deployed endpoints, enabling impact analysis when feature
   computations change.

---

## Cleanup

Experiments and lineage entities are metadata only -- they incur no compute
cost. However, it is good practice to remove them when they are no longer
needed to keep the console organized.

**What to clean up:**
- The Experiment and all its Runs (including the custom-metrics-demo Run)
- The reproducibility report in S3
- Notebook kernels (Studio > Running Terminals and Kernels > Shut down all)

**What to keep (if needed for later modules):**
- Training job records (cannot be deleted; they age out automatically)
- Auto-created lineage entities (managed by SageMaker)

**No endpoints were deployed** in this notebook, so there is nothing to delete
in the Endpoints console.

In [ ]:
try:
    experiment = Experiment.load(
        experiment_name=EXPERIMENT_NAME,
        sagemaker_session=sm_session,
    )
    experiment._delete_all(action="--force")
    print(f"Deleted experiment and all runs: {EXPERIMENT_NAME}")
except Exception as e:
    print(f"Cleanup note: {e}")
    print("The experiment may have already been deleted or may not exist.")

try:
    s3_client.delete_object(Bucket=default_bucket, Key=report_s3_key)
    print(f"Deleted report from S3: {report_s3_uri}")
except Exception as e:
    print(f"Report cleanup note: {e}")

In [ ]:
running_jobs = sm_client.list_training_jobs(
    StatusEquals="InProgress",
    MaxResults=10,
)

if running_jobs["TrainingJobSummaries"]:
    print("WARNING: The following training jobs are still running:")
    for job in running_jobs["TrainingJobSummaries"]:
        print(f"  {job['TrainingJobName']} -- {job['TrainingJobStatus']}")
    print()
    print("Stop them manually if they are from this lecture.")
else:
    print("No running training jobs found.")

print()
print("Remaining cleanup steps (manual):")
print("  1. Shut down all notebook kernels in Studio")
print("     (Studio > Running Terminals and Kernels > Shut Down All)")
print("  2. Verify no running training jobs in the Training Jobs console")
print("  3. Confirm no active endpoints (none were deployed in this notebook)")